In [ ]:
pip install openmeteo-requests

In [ ]:
pip install requests-cache retry-requests numpy pandas

In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://air-quality-api.open-meteo.com/v1/air-quality"

locations = [
    {"name": "Ambon City", "latitude": -3.6958,      "longitude": 128.1833},
    {"name": "Bali", "latitude": -8.3333, "longitude": 115},
    {"name": "Balikpapan", "latitude": -1.2674, "longitude": 116.8310},
    {"name": "Bandung", "latitude": -6.9222, "longitude": 107.6069},
    {"name": "Banda Aceh", "latitude": 5.5417,  "longitude": 95.3333},
    {"name": "Bekasi", "latitude": -6.2349, "longitude": 106.9896},
    {"name": "Gorontalo", "latitude": 0.5375,  "longitude": 123.0625},
    {"name": "Jakarta", "latitude": -6.2088, "longitude": 106.8456},
    {"name": "Jayapura", "latitude": -2.5337,   "longitude": 140.7181},
    {"name": "Jogja", "latitude": -3.4292, "longitude": 119.222},
    {"name": "Kendari", "latitude": -3.9778,   "longitude": 122.5151},
    {"name": "Kupang", "latitude": -10.1708,    "longitude": 123.6069},
    {"name": "Makassar", "latitude": 0.9039, "longitude": 122.7586},
    {"name": "Manado", "latitude": 1.4822,      "longitude": 124.8489},
    {"name": "Mataram", "latitude": -8.5833,   "longitude": 116.1167},
    {"name": "Medan", "latitude": 3.5833, "longitude": 98.6667},
    {"name": "Merauke", "latitude": -8.4996,     "longitude": 140.4061},
    {"name": "Padang", "latitude": -0.9492, "longitude": 100.3543},
    {"name": "Palangkaraya", "latitude": -2.2083,"longitude": 113.9165},
    {"name": "Palembang", "latitude": -2.9167, "longitude": 104.7458},
    {"name": "Palu", "latitude": -0.9083,       "longitude": 119.8708},
    {"name": "Pekanbaru", "latitude": 0.5167, "longitude": 101.4417},
    {"name": "Pontianak", "latitude": -0.0319,   "longitude": 109.325},
    {"name": "Samarinda", "latitude": -0.4917,   "longitude": 117.1458},
    {"name": "Semarang", "latitude": -6.9931, "longitude": 110.4208},
    {"name": "Surabaya", "latitude": -7.2575, "longitude": 112.7521},
    {"name": "Tarakan", "latitude": 3.3133,      "longitude": 117.5915},
    {"name": "Tegal", "latitude": -6.8694, "longitude": 109.1402},
    {"name": "Ternate", "latitude": 0.7906,      "longitude": 127.3842},
    {"name": "Wonosobo", "latitude": -7.3589, "longitude": 109.9031}
]

all_hourly_dataframes = []

for location in locations:
    print(f"Fetching data for {location['name']}...")
    params = {
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "hourly": ["us_aqi", "us_aqi_pm2_5", "us_aqi_nitrogen_dioxide", "us_aqi_carbon_monoxide", "us_aqi_ozone", "us_aqi_pm10", "us_aqi_sulphur_dioxide"],
        "timezone": "auto",
        "start_date": "2025-09-10",
        "end_date": "2025-10-10",
    }
    responses = openmeteo.weather_api(url, params=params)

    # Process first location. Add a for-loop for multiple locations or weather models
    response = responses[0]
    print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevation: {response.Elevation()} m asl")
    print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
    print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_us_aqi = hourly.Variables(0).ValuesAsNumpy()
    hourly_us_aqi_pm2_5 = hourly.Variables(1).ValuesAsNumpy()
    hourly_us_aqi_nitrogen_dioxide = hourly.Variables(2).ValuesAsNumpy()
    hourly_us_aqi_carbon_monoxide = hourly.Variables(3).ValuesAsNumpy()
    hourly_us_aqi_ozone = hourly.Variables(4).ValuesAsNumpy()
    hourly_us_aqi_pm10 = hourly.Variables(5).ValuesAsNumpy()
    hourly_us_aqi_sulphur_dioxide = hourly.Variables(6).ValuesAsNumpy()

    hourly_data = {
        "date": pd.date_range(
            start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
            end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
            freq = pd.Timedelta(seconds = hourly.Interval()),
            inclusive = "left"
        )
    }

    hourly_data["us_aqi"] = hourly_us_aqi
    hourly_data["us_aqi_pm2_5"] = hourly_us_aqi_pm2_5
    hourly_data["us_aqi_nitrogen_dioxide"] = hourly_us_aqi_nitrogen_dioxide
    hourly_data["us_aqi_carbon_monoxide"] = hourly_us_aqi_carbon_monoxide
    hourly_data["us_aqi_ozone"] = hourly_us_aqi_ozone
    hourly_data["us_aqi_pm10"] = hourly_us_aqi_pm10
    hourly_data["us_aqi_sulphur_dioxide"] = hourly_us_aqi_sulphur_dioxide

    hourly_dataframe_single_location = pd.DataFrame(data = hourly_data)
    hourly_dataframe_single_location['location_name'] = location['name']
    all_hourly_dataframes.append(hourly_dataframe_single_location)

# Concatenate all dataframes into a single one
overall_hourly_dataframe = pd.concat(all_hourly_dataframes, ignore_index=True)
print("\nOverall Hourly Data for all locations:\n")
display(overall_hourly_dataframe.head())
display(overall_hourly_dataframe.tail())


Fetching data for Ambon City...
Coordinates: -3.6999969482421875°N 128.20001220703125°E
Elevation: 7.0 m asl
Timezone: b'Asia/Jayapura'b'GMT+9'
Timezone difference to GMT+0: 32400s
Fetching data for Bali...
Coordinates: -8.299995422363281°N 115.0°E
Elevation: 616.0 m asl
Timezone: b'Asia/Makassar'b'GMT+8'
Timezone difference to GMT+0: 28800s
Fetching data for Balikpapan...
Coordinates: -1.2999954223632812°N 116.80001831054688°E
Elevation: 50.0 m asl
Timezone: b'Asia/Makassar'b'GMT+8'
Timezone difference to GMT+0: 28800s
Fetching data for Bandung...
Coordinates: -6.900001525878906°N 107.60000610351562°E
Elevation: 707.0 m asl
Timezone: b'Asia/Jakarta'b'GMT+7'
Timezone difference to GMT+0: 25200s
Fetching data for Banda Aceh...
Coordinates: 5.5°N 95.30001831054688°E
Elevation: 5.0 m asl
Timezone: b'Asia/Jakarta'b'GMT+7'
Timezone difference to GMT+0: 25200s
Fetching data for Bekasi...
Coordinates: -6.1999969482421875°N 107.0°E
Elevation: 19.0 m asl
Timezone: b'Asia/Jakarta'b'GMT+7'
Timezo

,date,us_aqi,us_aqi_pm2_5,us_aqi_nitrogen_dioxide,us_aqi_carbon_monoxide,us_aqi_ozone,us_aqi_pm10,us_aqi_sulphur_dioxide,location_name
0,2025-09-09 15:00:00+00:00,15.596010,11.458334,0.591017,5.560386,15.596010,3.087121,0.218103,Ambon City
1,2025-09-09 16:00:00+00:00,15.132191,11.354167,0.492514,6.044686,15.132191,3.075758,0.218103,Ambon City
2,2025-09-09 17:00:00+00:00,14.726344,11.267361,0.443262,6.349034,14.726344,3.071970,0.163577,Ambon City
3,2025-09-09 18:00:00+00:00,14.436457,11.180555,0.344760,6.411836,14.436457,3.068182,0.163577,Ambon City
4,2025-09-09 19:00:00+00:00,14.320501,11.076390,0.295508,6.138889,14.320501,3.064394,0.163577,Ambon City


,date,us_aqi,us_aqi_pm2_5,us_aqi_nitrogen_dioxide,us_aqi_carbon_monoxide,us_aqi_ozone,us_aqi_pm10,us_aqi_sulphur_dioxide,location_name
22315,2025-10-10 12:00:00+00:00,156.024124,156.024124,7.978724,2.729469,85.416664,56.758331,3.162487,Wonosobo
22316,2025-10-10 13:00:00+00:00,155.186401,155.186401,8.224981,3.483092,75.000000,55.933338,2.998909,Wonosobo
22317,2025-10-10 14:00:00+00:00,153.921051,153.921051,8.618992,4.439614,61.819725,54.695839,2.726282,Wonosobo
22318,2025-10-10 15:00:00+00:00,151.925430,151.925430,9.013002,5.577294,48.875229,52.766670,2.399128,Wonosobo
22319,2025-10-10 16:00:00+00:00,149.447891,149.447891,9.259259,6.822464,43.773190,50.785416,1.853872,Wonosobo


In [ ]:
import numpy as np

# Create a new DataFrame with the desired structure
air_quality_data = pd.DataFrame({
    'Date': overall_hourly_dataframe['date'],
    'City': overall_hourly_dataframe['location_name'],
    'CO': overall_hourly_dataframe['us_aqi_carbon_monoxide'],
    'CO2': np.nan, # Fill with NaN as requested
    'NO2': overall_hourly_dataframe['us_aqi_nitrogen_dioxide'],
    'SO2': overall_hourly_dataframe['us_aqi_sulphur_dioxide'],
    'O3': overall_hourly_dataframe['us_aqi_ozone'],
    'PM2.5': overall_hourly_dataframe['us_aqi_pm2_5'],
    'PM10': overall_hourly_dataframe['us_aqi_pm10'],
    'AQI': overall_hourly_dataframe['us_aqi']
})

print("Restructured Air Quality Data:")
display(air_quality_data.head())
display(air_quality_data.tail())

Restructured Air Quality Data:


,Date,City,CO,CO2,NO2,SO2,O3,PM2.5,PM10,AQI
0,2025-09-09 15:00:00+00:00,Ambon City,5.560386,NaN,0.591017,0.218103,15.596010,11.458334,3.087121,15.596010
1,2025-09-09 16:00:00+00:00,Ambon City,6.044686,NaN,0.492514,0.218103,15.132191,11.354167,3.075758,15.132191
2,2025-09-09 17:00:00+00:00,Ambon City,6.349034,NaN,0.443262,0.163577,14.726344,11.267361,3.071970,14.726344
3,2025-09-09 18:00:00+00:00,Ambon City,6.411836,NaN,0.344760,0.163577,14.436457,11.180555,3.068182,14.436457
4,2025-09-09 19:00:00+00:00,Ambon City,6.138889,NaN,0.295508,0.163577,14.320501,11.076390,3.064394,14.320501


,Date,City,CO,CO2,NO2,SO2,O3,PM2.5,PM10,AQI
22315,2025-10-10 12:00:00+00:00,Wonosobo,2.729469,NaN,7.978724,3.162487,85.416664,156.024124,56.758331,156.024124
22316,2025-10-10 13:00:00+00:00,Wonosobo,3.483092,NaN,8.224981,2.998909,75.000000,155.186401,55.933338,155.186401
22317,2025-10-10 14:00:00+00:00,Wonosobo,4.439614,NaN,8.618992,2.726282,61.819725,153.921051,54.695839,153.921051
22318,2025-10-10 15:00:00+00:00,Wonosobo,5.577294,NaN,9.013002,2.399128,48.875229,151.925430,52.766670,151.925430
22319,2025-10-10 16:00:00+00:00,Wonosobo,6.822464,NaN,9.259259,1.853872,43.773190,149.447891,50.785416,149.447891


In [ ]:
air_quality_data.to_csv('air_quality_data(fetch).csv', index=False)
print("Dataset telah berhasil disimpan")

Dataset telah berhasil disimpan
